In [ ]:
from pyspark.sql.functions import (
    col, to_date, concat, lit, when, substring,
    year, month, current_timestamp, current_date, lpad
)
from pyspark.sql.types import IntegerType, DecimalType, DateType, StringType
from delta.tables import DeltaTable

CATALOG = "czech_fintech"
BRONZE  = "bronze"
SILVER  = "silver"

def parse_czech_date(c):
    """YYMMDD stringa → DATE. YY < 50 → 20YY, altrimenti 19YY."""
    return to_date(
        when(substring(c, 1, 2).cast(IntegerType()) < 50, concat(lit("20"), c))
        .otherwise(concat(lit("19"), c)),
        "yyyyMMdd"
    )

In [ ]:
# ============================================================================
# TRANSACTIONS — typing + join client_id via disp (OWNER)
# ============================================================================

disp_owner = (
    spark.table(f"{CATALOG}.{BRONZE}.disp")
    .filter(col("type") == "OWNER")
    .select("account_id", "client_id")
)

trans_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.transactions")
    .drop("_ingestion_ts", "_source_file")
    .join(disp_owner, "account_id", "left")
    .withColumn("date",    parse_czech_date(col("date")))
    .withColumn("amount",  col("amount").cast(DecimalType(15,2)))
    .withColumn("balance", col("balance").cast(DecimalType(15,2)))
    .withColumn("year",    year(col("date")))
    .withColumn("month",   month(col("date")))
    .withColumn("_silver_ts", current_timestamp())
)

trans_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("year", "month") \
    .saveAsTable(f"{CATALOG}.{SILVER}.transactions")

print(f"✅ transactions: {spark.table(f'{CATALOG}.{SILVER}.transactions').count()} rows")

In [ ]:
# ============================================================================
# CLIENT — typing + decodifica birth_number (gender, birth_date)
# ============================================================================

client_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.client")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("birth_month_raw", substring(col("birth_number"), 3, 2).cast(IntegerType()))
    .withColumn("gender",
        when(col("birth_month_raw") > 50, lit("F")).otherwise(lit("M"))
    )
    .withColumn("birth_month_norm",
        when(col("birth_month_raw") > 50, col("birth_month_raw") - 50)
        .otherwise(col("birth_month_raw"))
    )
    .withColumn("birth_date",
        to_date(
            concat(
                lit("19"), substring(col("birth_number"), 1, 2),
                lpad(col("birth_month_norm").cast(StringType()), 2, "0"),
                substring(col("birth_number"), 5, 2)
            ),
            "yyyyMMdd"
        )
    )
    .drop("birth_month_raw", "birth_month_norm")
    .withColumn("_silver_ts", current_timestamp())
)

client_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.client")

print(f"✅ client: {spark.table(f'{CATALOG}.{SILVER}.client').count()} rows")

In [ ]:
# ============================================================================
# LOAN — typing
# ============================================================================

loan_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.loan")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("date",     parse_czech_date(col("date")))
    .withColumn("amount",   col("amount").cast(DecimalType(15,2)))
    .withColumn("duration", col("duration").cast(IntegerType()))
    .withColumn("payments", col("payments").cast(DecimalType(15,2)))
    .withColumn("_silver_ts", current_timestamp())
)

loan_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.loan")

print(f"✅ loan: {spark.table(f'{CATALOG}.{SILVER}.loan').count()} rows")

# ============================================================================
# ORDER — typing
# ============================================================================

order_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.order")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("amount", col("amount").cast(DecimalType(15,2)))
    .withColumn("_silver_ts", current_timestamp())
)

order_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.order")

print(f"✅ order: {spark.table(f'{CATALOG}.{SILVER}.order').count()} rows")

# ============================================================================
# CARD — typing
# issued è nel formato "YYYYMMDD HH:MM:SS", non YYMMDD → prendo solo i primi 8 char
# ============================================================================

card_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.card")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("issued", to_date(substring(col("issued"), 1, 8), "yyyyMMdd"))
    .withColumn("_silver_ts", current_timestamp())
)

card_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.card")

print(f"✅ card: {spark.table(f'{CATALOG}.{SILVER}.card').count()} rows")

# ============================================================================
# DISP — passthrough
# ============================================================================

disp_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.disp")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("_silver_ts", current_timestamp())
)

disp_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.disp")

print(f"✅ disp: {spark.table(f'{CATALOG}.{SILVER}.disp').count()} rows")

# ============================================================================
# DISTRICT — passthrough
# ============================================================================

district_silver = (
    spark.table(f"{CATALOG}.{BRONZE}.district")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("_silver_ts", current_timestamp())
)

district_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{SILVER}.district")

print(f"✅ district: {spark.table(f'{CATALOG}.{SILVER}.district').count()} rows")

In [ ]:
# ============================================================================
# ACCOUNT — typing + SCD Type 2 su frequency
# ============================================================================

account_incoming = (
    spark.table(f"{CATALOG}.{BRONZE}.account")
    .drop("_ingestion_ts", "_source_file")
    .withColumn("date",       parse_czech_date(col("date")))
    .withColumn("valid_from", current_date())
    .withColumn("valid_to",   lit(None).cast(DateType()))
    .withColumn("is_current", lit(True))
    .withColumn("_silver_ts", current_timestamp())
)

if spark.catalog.tableExists(f"{CATALOG}.{SILVER}.account"):
    dt = DeltaTable.forName(spark, f"{CATALOG}.{SILVER}.account")

    # Chiudi righe correnti dove frequency è cambiata
    dt.alias("existing").merge(
        account_incoming.alias("incoming"),
        "existing.account_id = incoming.account_id AND existing.is_current = true"
    ).whenMatchedUpdate(
        condition="existing.frequency != incoming.frequency",
        set={"valid_to": "current_date()", "is_current": "false"}
    ).execute()

    # Inserisci solo gli account che non hanno già una riga corrente identica
    existing_current = spark.table(f"{CATALOG}.{SILVER}.account").filter(col("is_current"))
    new_rows = account_incoming.join(existing_current.select("account_id", "frequency"),
                                     ["account_id", "frequency"], "left_anti")
    new_rows.write.mode("append").saveAsTable(f"{CATALOG}.{SILVER}.account")
else:
    account_incoming.write.mode("overwrite").option("overwriteSchema", "true") \
        .saveAsTable(f"{CATALOG}.{SILVER}.account")

print(f"✅ account: {spark.table(f'{CATALOG}.{SILVER}.account').count()} rows")